/tmp/ipykernel_22968/3525928394.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(
/tmp/ipykernel_22968/3525928394.py:41: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [ ]:
from langchain.agents import initialize_agent, Tool, AgentType
from GeminiLoader import llm

agent = initialize_agent(
    tools=[
        Tool(
            name="contract_search",
            func=search_contract_doc,
            description="useful for when you need to answer questions about the contract"
        )
    ],
    llm=llm,
    agent_type=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)


prompt = """
You are a Contract assistant. You have access to a contract search tool.

Steps:
1. You have to extract information from contract document using the contract_search tool by constructing query for similarity search.
2. The information you need to extract is the following:
- Contract Type
- Contract Date
- Parties Involved (Organization Names)
- Summary of the contract
3. If you are not able to extract the information, you can say "I don't know".
    (Thought/Observation/Action can be repeated N times)
4. If you are able to answer, then format it in the following schema:
{
    "Contract Type": "<contract_type>",
    "Contract Date": "<contract_date>",
    "Parties Involved": "<parties_involved>",
    "Summary": "<summary>"
}
"""

resp = agent.run(prompt)

/tmp/ipykernel_22968/2046504762.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
/tmp/ipykernel_22968/2046504762.py:41: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  resp = agent.run(prompt)




> Entering new AgentExecutor chain...
Okay, I understand the task. I need to use the `contract_search` tool to extract information about the contract, including its type, name, date, parties involved, and a summary. I'll start by searching for the contract type.

Action: contract_search
Action Input: "Contract Type"
Observation: [Document(metadata={'trapped': '', 'total_pages': 24, 'page': 19, 'modDate': "D:20200708220212-07'00'", 'title': '', 'keywords': '', 'creationDate': '', 'file_path': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'creationdate': '', 'moddate': '2020-07-08T22:02:12-07:00', 'author': '', 'subject': '', 'format': 'PDF 1.4', 'source': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'producer': 'Aspose.PDF for .NET 18.4', 'creator': 'Aspose Ltd.'}, page_content='majeure continues and the nonperforming party takes reasonable efforts to remove or circumvent the interference caused by the condition. For\npu

/tmp/ipykernel_22968/1746373905.py:15: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return retriever.get_relevant_documents(query)


Okay, the search results mention "Supply Agreement". Let's see if I can confirm the Contract Type and get the Contract Name.

Action: contract_search
Action Input: "Supply Agreement"
Observation: [Document(metadata={'page': 5, 'moddate': '2020-07-08T22:02:12-07:00', 'total_pages': 24, 'author': '', 'keywords': '', 'file_path': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'title': '', 'trapped': '', 'producer': 'Aspose.PDF for .NET 18.4', 'modDate': "D:20200708220212-07'00'", 'format': 'PDF 1.4', 'creationdate': '', 'subject': '', 'creator': 'Aspose Ltd.', 'source': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'creationDate': ''}, page_content='2.2. Fees.\n(a) Treated Integra Products. Subject to Section 2.2(c), for so long as the Agreement has not been converted to a non-exclusive arrangement\nunder the provisions of Section 3.2, Integra shall pay PcoMed a Fee of ***% of Net Sales of all Treated Integra Product Sold by I

# Looper


In [1]:
import spacy
import re,json
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
import shutil
import os

def get_pdf_filenames():
    """
    Get a list of all PDF file names in the 'pdfs' folder located in the current working directory.

    Returns:
        list: List of PDF file names.
    """
    cwd = os.getcwd()
    pdfs_folder = os.path.join(cwd, 'contract_pdfs')
    return [file for file in os.listdir(pdfs_folder) if file.endswith('.pdf')]

def delete_persist_db(pdf_name):
    """
    Deletes the persistent database for the given PDF file.

    Args:
        pdf_name (str): The name of the PDF file whose database to delete.
    """
    db_path = f"db/{pdf_name.split('_')[0]}"
    if os.path.exists(db_path):
        shutil.rmtree(db_path) # Uncomment if you want to delete the file instead
        print(f"Deleted {db_path}")
    else:
        print(f"{db_path} does not exist.")

def process_pdf_to_retriever(pdf_name):
    """
    Processes a PDF file to create a retriever from its embedded vectorstore.

    Args:
        pdf_name (str): The name of the PDF file to process.

    Returns:
        ChromaRetriever: A retriever object for the processed PDF.
    """
    loader = PyMuPDFLoader(f'contract_pdfs/{pdf_name}')
    docs = loader.load()

    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    splitted_docs = splitter.split_documents(docs)

    embeddings = SentenceTransformerEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cuda"}
    )
        

    vectorstore = Chroma.from_documents(
        splitted_docs,
        embeddings,
        collection_name=pdf_name.split('_')[0],
        persist_directory=f"db/{pdf_name.split('_')[0]}",
    )

    vectorstore.persist()
    return vectorstore, splitted_docs
    



def clean_response(resp):
    decoded = bytes(resp, "utf-8").decode("unicode_escape")
    # Step 2: Remove markdown code block syntax
    cleaned = re.sub(r'```json|```', '', decoded).strip().strip('"')

    # Step 3: Load JSON
    return json.loads(cleaned)

def extract_named_entities(splitted_docs):
    nlp = spacy.load("en_core_web_lg")
    ner = []
    # Process spacy to get names, nouns, dates, and organizations
    entities = {
        "names": [],
        "dates": [],
        "organizations": [],
        "numerical_values": []
    }
    for doc in splitted_docs:
        doc_analysis = nlp(doc.page_content)
        entities["names"] += [ent.text for ent in doc_analysis.ents if ent.label_ == "PERSON" and ent.text not in entities["names"]]
        entities["dates"] += [ent.text for ent in doc_analysis.ents if ent.label_ == "DATE" and ent.text not in entities["dates"]]
        entities["organizations"] += [ent.text for ent in doc_analysis.ents if ent.label_ == "ORG" and ent.text not in entities["organizations"]]
        entities["numerical_values"] += [token.text for token in doc_analysis if token.pos_ == "NUM" and token.text not in entities["numerical_values"]]

    return entities

def get_Prompt():
    return """
    You are a Contract assistant. You have access to a contract search tool.

    Steps:
    1. You have to extract information from contract document using the contract_search tool by constructing query for similarity search.
    2. The information you need to extract is the following:
    - Contract Type
    - Contract Date
    - Parties Involved (Organization Names List)
    - Summary of the contract
    3. If you are not able to extract the information, you can say "I don't know".
        (Thought/Observation/Action can be repeated N times)
    4. If you are able to answer, then format it in the following schema:
    {
        "Contract Type": "<contract_type>",
        "Contract Date": "<contract_date>",
        "Parties Involved": ["<list_of_parties_involved>"],
        "Summary": "<summary>"
    }
    """

In [2]:
from langchain.agents import initialize_agent, Tool, AgentType
from GeminiLoader import llm

doc_objs = []
pdf_files = get_pdf_filenames()
prompt = get_Prompt()
for file in pdf_files:
    try:
        delete_persist_db(file)
        print("Flag1")
        vc, splitted_docs = process_pdf_to_retriever(file)
        print("Flag2")
        

        def search_contract_doc(query):
            """
            Searches for relevant documents in the vectorstore based on the provided query.

            Args:
                query (str): The search query.

            Returns:
                list: List of relevant documents.
            """
            retriever = vc.as_retriever(search_kwargs={"k": 3})
            return retriever.get_relevant_documents(query)
        print("Flag3 Running agent")
        
        agent = initialize_agent(
            tools=[
                Tool(
                    name="contract_search",
                    func=search_contract_doc,
                    description="useful for when you need to answer questions about the contract"
                )
            ],
            llm=llm,
            agent_type=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
            verbose=True,
            handle_parsing_errors=True
        )
    
        resp = agent.run(prompt)
        
        doc_obj = clean_response(resp)
        
        doc_obj['file_name'] = file.split('_')[0]
        
        NERs = extract_named_entities(splitted_docs)
        doc_obj['NERs'] = NERs
        
        doc_objs.append(doc_obj)
        
        del vc
        del agent
    except Exception as e:
        print(e)
        print("Resource Exhausted")
        print(doc_objs)
        break
    
    
doc_objs

Deleted db/SEASPINEHOLDINGSCORP
Flag1


/tmp/ipykernel_24848/1269165035.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(
/tmp/ipykernel_24848/1269165035.py:69: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()
/tmp/ipykernel_24848/2415446110.py:29: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and hum

Flag2
Flag3 Running agent


> Entering new AgentExecutor chain...
Okay, I understand the task. I need to extract information about the contract type, date, parties involved, and a summary. I will use the `contract_search` tool to find this information. I will start by searching for the contract type.
Action: contract_search
Action Input: "contract type"
Observation: [Document(metadata={'creator': 'Aspose Ltd.', 'page': 19, 'file_path': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'author': '', 'format': 'PDF 1.4', 'total_pages': 24, 'modDate': "D:20200708220212-07'00'", 'source': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'moddate': '2020-07-08T22:02:12-07:00', 'keywords': '', 'subject': '', 'producer': 'Aspose.PDF for .NET 18.4', 'trapped': '', 'creationdate': '', 'title': '', 'creationDate': ''}, page_content='majeure continues and the nonperforming party takes reasonable efforts to remove or circumvent the interfere

/tmp/ipykernel_24848/2415446110.py:26: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return retriever.get_relevant_documents(query)


Okay, the search results mention "Supply Agreement". Let's see if I can confirm this and get the contract date.
Action: contract_search
Action Input: "Supply Agreement date"
Observation: [Document(metadata={'modDate': "D:20200708220212-07'00'", 'author': '', 'creationDate': '', 'producer': 'Aspose.PDF for .NET 18.4', 'page': 23, 'format': 'PDF 1.4', 'moddate': '2020-07-08T22:02:12-07:00', 'source': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf', 'creationdate': '', 'title': '', 'trapped': '', 'total_pages': 24, 'creator': 'Aspose Ltd.', 'subject': '', 'keywords': '', 'file_path': 'contract_pdfs/SEASPINEHOLDINGSCORP_10_10_2018-EX-10.1-SUPPLY AGREEMENT.pdf'}, page_content='ATTACHMENT C\nNOTICE OF INITIAL ACCEPTANCE OF FIRST PRODUCT ORDER\nThis Notice references the Agreement executed effective as of , 2013, by and between Integra LifeSciences Corporation (“Integra”) and PcoMed, LLC\n(“PcoMed”) (“Agreement”). In accordance with Section 12.1 (Notices) and purs

...

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
}
].


429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 57
}
]
Resource Exhausted
[{'Contract Type': 'Supply Agreement', 'Contract Date': 'May 29, 2013 (Integra) and May 15, 2013 (PcoMed)', 'Parties Involved': ['Integra LifeSciences Corporation', 'PCOMED, LLC'], 'Summary': 'This is a Supply Agreement between Integra LifeSciences Corporation and PCOMED, LLC. PcoMed grants Integra an exclusive worldwid

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 1
}
].


429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 58
}
]
Resource Exhausted
[{'Contract Type': 'Supply Agreement', 'Contract Date': 'May 29, 2013 (Integra) and May 15, 2013 (PcoMed)', 'Parties Involved': ['Integra LifeSciences Corporation', 'PCOMED, LLC'], 'Summary': 'This is a Supply Agreement between Integra LifeSciences Corporation and PCOMED, LLC. PcoMed grants Integra an exclusive worldwid

[{'Contract Type': 'Supply Agreement',
  'Contract Date': 'May 29, 2013 (Integra) and May 15, 2013 (PcoMed)',
  'Parties Involved': ['Integra LifeSciences Corporation', 'PCOMED, LLC'],
  'Summary': 'This is a Supply Agreement between Integra LifeSciences Corporation and PCOMED, LLC. PcoMed grants Integra an exclusive worldwide right to sell and commercialize Integra Products treated by PcoMed, with the PcoMed Surface Modification Technology for use in the Field in the Territory.',
  'file_name': 'SEASPINEHOLDINGSCORP',
  'NERs': {'names': ['Theken Spine',
    'Attachment B.\n3',
    'Brian Larkin',
    'Patricia Jacobson',
    'Steve Jacobs',
    'Alan Keeffe',
    'Steven Jacobs',
    'Steve Jacobs\nName'],
   'dates': ['1934',
    'this 15th day of May, 2013',
    '08536',
    'the thirty six (36) month period',
    'sixty (60) days',
    '30 days',
    'sixty (60) days',
    'quarterly',
    'thirty (30) days',
    'annually',
    '45 days',
    'the last day',
    '45) days',
    '